# Carbon utilization KEGG pathways

https://github.com/zeqianli/CarbonUtilization

In [1]:
%load_ext autoreload
%autoreload 2

import os, re, requests
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 15})
plt.rcParams['figure.dpi'] = 200
import seaborn as sns
import pickle
import itertools
import random
import traceback
import networkx as nx
import scipy.cluster.hierarchy as hierarchy
from functools import reduce
from tqdm import tqdm
from multiprocessing import Pool
from ete3 import Tree
from Bio import Phylo
from io import StringIO
from scipy.stats import ttest_ind
from statannotations.Annotator import Annotator
import sklearn.metrics as metrics 
from scipy.stats import entropy
from statsmodels.stats.multitest import multipletests

from kegg_utils import Kegg

# Scrape KEGG data

In [2]:
DIR_kegg="/home/acs98/Documents/BCEM/Tesis-maestria/masters-thesis/metabolic-modeling/CarbonUtilization/kegg"

### Pathways

In [3]:
ignore_pathways = [
    'map01100', # Metabolic pathways
    'map01110', # Biosynthesis of secondary metabolites
    'map01120', # Microbial metabolism in diverse environments
    'map01200', # Carbon metabolism
    'map01210', # 2-Oxocarboxylic acid metabolism
    'map01212', # Fatty acid metabolism
    'map01230', # Biosynthesis of amino acids
    'map01232', # Nucleotide metabolism
    'map01250', # Biosynthesis of nucleotide sugars
    'map01240', # Biosynthesis of cofactors
    'map01220', # Degradation of aromatic compounds
    'map00195', # Photosynthesis
    'map00196', # Photosynthesis - antenna proteins
    'map00710', # Carbon fixation in photosynthetic organisms
    'map00910', # Nitrogen metabolism
    'map00920', # Sulfur metabolism
    'map01010', # Overview of biosynthetic pathways
    'map01060', # Biosynthesis of plant secondary metabolites
    'map01061', # Biosynthesis of phenylpropanoids
    'map01062', # Biosynthesis of terpenoids and steroids
    'map01063', # Biosynthesis of alkaloids derived from shikimate pathway
    'map01064', # Biosynthesis of alkaloids derived from ornithine, lysine and nicotinic acid
    'map01065', # Biosynthesis of alkaloids derived from histidine and purine
    'map01066', # Biosynthesis of alkaloids derived from terpenoid and polyketide
    'map01070', # Biosynthesis of plant hormones
    'map01501', # beta-Lactam resistance
    'map01502', # Vancomycin resistance
    'map01503', # Cationic antimicrobial peptide (CAMP) resistance
    'map01521', # EGFR tyrosine kinase inhibitor resistance
    'map01522', # Endocrine resistance
    'map01523', # Antifolate resistance
    'map01524', # Platinum drug resistance
]
to_focus = [
    'map01200', # Carbon metabolism
    'map01210', # 2-Oxocarboxylic acid metabolism
    'map01212', # Fatty acid metabolism
    'map01230', # Biosynthesis of amino acids
    'map01232', # Nucleotide metabolism
]

In [4]:
# All metabolism pathways. Pathway id starts with "map00" or "map01"
# https://www.genome.jp/kegg/pathway.html
pathways = pd.read_csv("https://rest.kegg.jp/list/pathway",sep='\t+',header=None,names=["pathway_id","pathway_name"],engine='python')
pathways=pathways[pathways.pathway_id.str.contains("map00|map01")]
# pathways=pathways[~(pathways['pathway_id'].isin(ignore_pathways))] # overview maps 
pathways=pathways[(pathways['pathway_id'].isin(to_focus))] # overview maps 
pathways_ = pathways.copy()
pathways_ids=pathways['pathway_id'].values
pathways
# archive
# pathways=pd.read_csv(os.path.join(DIR_kegg,"pathway_list.txt"),header=None).values.flatten()

,pathway_id,pathway_name
3,map01200,Carbon metabolism
4,map01210,2-Oxocarboxylic acid metabolism
5,map01212,Fatty acid metabolism
6,map01230,Biosynthesis of amino acids
7,map01232,Nucleotide metabolism


In [5]:
# make directories
for key in ['kos','pathways','compounds','modules','reactions','graphs']:
    if not os.path.exists(os.path.join(DIR_kegg,key)):
        os.makedirs(os.path.join(DIR_kegg,key))

In [5]:
# kos 
kos=Kegg.download_csv(["link/ko",*pathways_ids],labels=['pathway','ko'])
kos['pathway']=kos['pathway'].str.replace("path:","")
kos['ko']=kos['ko'].str.replace("ko:","")

for _, pathway, ko in tqdm(list(kos.itertuples())):
    if not os.path.exists(os.path.join(DIR_kegg,"kos",ko)):
        with open(os.path.join(DIR_kegg,"kos",ko),'w') as f:
            pass # create an empty file
    
    if not os.path.exists(os.path.join(DIR_kegg,"pathways",pathway)):
        os.makedirs(os.path.join(DIR_kegg,"pathways",pathway))
    with open(os.path.join(DIR_kegg,"pathways",pathway,'kos.txt'),'a') as f:
        f.write(ko)
        f.write('\n')
kos

0it [00:00, ?it/s]

1it [00:01,  1.94s/it]
100%|██████████| 924/924 [00:00<00:00, 22282.81it/s]


,pathway,ko
0,map01200,K00018
1,map01200,K00023
2,map01200,K00024
3,map01200,K00025
4,map01200,K00026
...,...,...
919,map01232,K25589
920,map01232,K25641
921,map01232,K26110
922,map01232,K26211


In [6]:
# reactions
reactions=Kegg.download_csv(["link/reaction",*pathways_ids],labels=['pathway','reaction'])
reactions['pathway']=reactions['pathway'].str.replace("path:","")
reactions['reaction']=reactions['reaction'].str.replace("rn:","")

for _, pathway, reaction in reactions.itertuples():
    if not os.path.exists(os.path.join(DIR_kegg,"reactions",reaction)):
        os.makedirs(os.path.join(DIR_kegg,"reactions",reaction))
    
    if not os.path.exists(os.path.join(DIR_kegg,"pathways",pathway)):
        os.makedirs(os.path.join(DIR_kegg,"pathways",pathway))
    with open(os.path.join(DIR_kegg,"pathways",pathway,'reactions.txt'),'a') as f:
        f.write(reaction)
        f.write('\n')
reactions

0it [00:00, ?it/s]

1it [00:02,  2.14s/it]


,pathway,reaction
0,map01200,R00009
1,map01200,R00024
2,map01200,R00134
3,map01200,R00149
4,map01200,R00150
...,...,...
758,map01232,R11636
759,map01232,R12589
760,map01232,R12955
761,map01232,R12956


In [7]:
# compounds
compounds=Kegg.download_csv(["link/compound",*pathways_ids],labels=['pathway','compound'])
compounds['pathway']=compounds['pathway'].str.replace("path:","")
compounds['compound']=compounds['compound'].str.replace("cpd:","")

for _, pathway, compound in compounds.itertuples():
    if not os.path.exists(os.path.join(DIR_kegg,"compounds",compound)):
        with open(os.path.join(DIR_kegg,"compounds",compound),'w') as f:
            pass # create an empty file
    
    if not os.path.exists(os.path.join(DIR_kegg,"pathways",pathway)):
        os.makedirs(os.path.join(DIR_kegg,"pathways",pathway))
    with open(os.path.join(DIR_kegg,"pathways",pathway,'compounds.txt'),'a') as f:
        f.write(compound)
        f.write('\n')
compounds

0it [00:00, ?it/s]

1it [00:02,  2.03s/it]


,pathway,compound
0,map01200,C00011
1,map01200,C00014
2,map01200,C00022
3,map01200,C00024
4,map01200,C00025
...,...,...
559,map01232,C01346
560,map01232,C01762
561,map01232,C03794
562,map01232,C05512


In [8]:
# compounds_with_pathway_names=compounds.merge(pathways,left_on='pathway',right_on='pathway_id')
# compounds_with_pathway_names.drop(columns=['pathway'],inplace=True)
# compounds_with_pathway_names.to_csv(os.path.join(DIR_kegg,"compounds_with_pathway_names.csv"),index=False)
# compounds_with_pathway_names
compounds_with_pathway_names=compounds.merge(pathways,left_on='pathway',right_on='pathway_id')
compounds_with_pathway_names.drop(columns=['pathway'],inplace=True)
compounds_with_pathway_names.to_csv(os.path.join(DIR_kegg,"compounds_with_pathway_names_int.csv"),index=False)
compounds_with_pathway_names

,compound,pathway_id,pathway_name
0,C00011,map01200,Carbon metabolism
1,C00014,map01200,Carbon metabolism
2,C00022,map01200,Carbon metabolism
3,C00024,map01200,Carbon metabolism
4,C00025,map01200,Carbon metabolism
...,...,...,...
559,C01346,map01232,Nucleotide metabolism
560,C01762,map01232,Nucleotide metabolism
561,C03794,map01232,Nucleotide metabolism
562,C05512,map01232,Nucleotide metabolism


### Reactions

In [11]:
reactions=os.listdir(os.path.join(DIR_kegg,"reactions"))

compounds=Kegg.download_csv(["link/compound",*reactions],labels=['reaction','compound'])
compounds['reaction']=compounds['reaction'].str.replace("rn:","")
compounds['compound']=compounds['compound'].str.replace("cpd:","")

for _, reaction, compound in compounds.itertuples():
    with open(os.path.join(DIR_kegg,"reactions",reaction,'compounds.txt'),'a') as f:
        f.write(compound)
        f.write('\n')

kos=Kegg.download_csv(["link/ko",*reactions],labels=['reaction','ko'])
kos['reaction']=kos['reaction'].str.replace("rn:","")
kos['ko']=kos['ko'].str.replace("ko:","")

for _, reaction, ko in kos.itertuples():
    with open(os.path.join(DIR_kegg,"reactions",reaction,'kos.txt'),'a') as f:
        f.write(ko)
        f.write('\n')

787it [27:31,  2.10s/it]
787it [28:47,  2.20s/it]


### Full text

In [12]:
# pathways

pathways=os.listdir(os.path.join(DIR_kegg,"pathways"))
for pathway in tqdm(pathways,desc="Retriving pathway information"):
    try:
        res=requests.get(f"http://rest.kegg.jp/get/{pathway}")
        if not bool(res):
            raise ValueError
        with open(os.path.join(DIR_kegg,"pathways",pathway,pathway),'w') as f:
            f.write(res.text)
    except Exception as e:
        print("Failed: ",pathway)
        continue

Retriving pathway information:   0%|          | 0/185 [00:00<?, ?it/s]

Retriving pathway information: 100%|██████████| 185/185 [06:17<00:00,  2.04s/it]


In [13]:
# reactions

reactions=os.listdir(os.path.join(DIR_kegg,"reactions"))
for reaction in tqdm(reactions, desc="Retriving reaction information"):
    try:
        res=requests.get(f"http://rest.kegg.jp/get/{reaction}")
        if not bool(res):
            raise ValueError
        with open(os.path.join(DIR_kegg,"reactions",reaction,reaction),'w') as f:
            f.write(res.text)
    except Exception as e:
        print("Failed: ",reaction)
        continue

Retriving reaction information:   0%|          | 0/7870 [00:00<?, ?it/s]

Retriving reaction information: 100%|██████████| 7870/7870 [4:02:29<00:00,  1.85s/it]  


In [14]:
# kos

kos=os.listdir(os.path.join(DIR_kegg,"kos"))
for ko in tqdm(kos, desc="Retriving ko information"):
    try:
        res=requests.get(f"http://rest.kegg.jp/get/{ko}")
        if not bool(res):
            raise ValueError
        with open(os.path.join(DIR_kegg,"kos",ko),'w') as f:
            f.write(res.text)
    except Exception as e:
        print("Failed: ",ko)
        continue

Retriving ko information:   0%|          | 0/6565 [00:00<?, ?it/s]

Retriving ko information: 100%|██████████| 6565/6565 [3:39:19<00:00,  2.00s/it]  


### Parse reaction graph

In [5]:
# From reaction texts. Use the EQUATION field

def parse_reaction(text):
    text=re.findall(r"EQUATION\s+(.+)\n",text)[0].strip()
    try:
        lefts,rights=text.split("<=>")
    except ValueError:
        try:
            lefts,rights=text.split("=>")
        except ValueError:
            rights,lefts=text.split("<=")
    lefts=lefts.strip()
    lefts=[re.findall('C\d+',l)[0] for l in lefts.split("+") if re.findall('C\d+',l)]
    rights=rights.strip()
    rights=[re.findall('C\d+',r)[0] for r in rights.split("+") if re.findall('C\d+',r)]
    return lefts,rights



In [6]:
# reaction graph 

with open(os.path.join(DIR_kegg,"graphs", 'reaction_graph_raw.txt'),'w') as f_graph:
    for reaction in tqdm(os.listdir(os.path.join(DIR_kegg,'reactions')), desc="Building reaction graph..."):
        try:
            with open(os.path.join(DIR_kegg,"reactions",reaction,reaction),'r') as f:
                text=f.read()
            lefts, rights=parse_reaction(text)
            for left in lefts:
                for right in rights:
                    f_graph.write(f"{reaction}\t{left}\t{right}\n")
        except Exception as e:
            print("Failed: ",reaction)
            print(e)
            continue

Building reaction graph...:   0%|          | 0/7870 [00:00<?, ?it/s]

Building reaction graph...: 100%|██████████| 7870/7870 [00:00<00:00, 31055.21it/s]


In [8]:
with open(os.path.join(DIR_kegg,'ignore_compounds.txt'),'r') as f:
    ignore_compounds=[re.findall(r'C\d+',line)[0] for line in f.readlines() if line.strip()]


In [9]:
edges=pd.read_csv(os.path.join(DIR_kegg, "graphs",'reaction_graph_raw.txt'), sep='\t', header=None,names=['reaction','c1','c2'])
edges=edges[(~edges['c1'].isin(ignore_compounds)) & (~edges['c2'].isin(ignore_compounds))]
edges=edges.drop_duplicates()
edges['c1_c2']=edges['c1']+'_'+edges['c2']
edges['c1_c2_undirected']=edges[['c1','c2']].apply(lambda x: '_'.join(sorted(x)), axis=1)

edges_directed=edges.groupby('c1_c2')['reaction'].apply(lambda x: ','.join(x)).reset_index()
edges_directed['c1']=edges_directed['c1_c2'].apply(lambda x: x.split('_')[0])
edges_directed['c2']=edges_directed['c1_c2'].apply(lambda x: x.split('_')[1])
edges_directed['n_reactions']=edges_directed['reaction'].apply(lambda x: len(x.split(',')))


edges_undirected=edges.groupby('c1_c2_undirected')['reaction'].apply(lambda x: ','.join(x)).reset_index()
edges_undirected['c1']=edges_undirected['c1_c2_undirected'].apply(lambda x: x.split('_')[0])
edges_undirected['c2']=edges_undirected['c1_c2_undirected'].apply(lambda x: x.split('_')[1])
edges_undirected['n_reactions']=edges_undirected['reaction'].apply(lambda x: len(x.split(',')))

g_directed=nx.from_pandas_edgelist(edges_directed, source='c1',target='c2',edge_attr=['reaction'],create_using=nx.DiGraph)

with open(os.path.join(DIR_kegg, "graphs",'reaction_graph_directed.pk'),'wb') as f:
    pickle.dump(g_directed,f)

g_undirected=nx.from_pandas_edgelist(edges_undirected, source='c1',target='c2',edge_attr=['reaction'],create_using=nx.Graph)

with open(os.path.join(DIR_kegg,"graphs",'reaction_graph_undirected.pk'),'wb') as f:
    pickle.dump(g_undirected,f)

#### Find top k paths

# Identify KOs related to a carbon compound

In [10]:
DIR_kegg="/home/acs98/Downloads/CarbonUtilization/kegg"

In [12]:
def find_reactions(g, source, target, k=1,remove_target_interlinks=True): # cutoff=100, 
    if isinstance(source, str):
        source=[source]
    if isinstance(target, str):
        target=[target]
    if remove_target_interlinks:
        g=g.copy()
        for t1, t2 in itertools.permutations(target,2):
            if g.has_edge(t1,t2):
                g.remove_edge(t1,t2)
    
    paths=[]
    for s in source:
        for t in target:
            try:
                for i, path in enumerate(nx.shortest_simple_paths(g, s, t)):
                    if i>=k:
                        break
                    paths.append(path)
            except nx.NodeNotFound:
                continue
            except nx.NetworkXNoPath:
                continue

    
    #print(paths)
    path_lengths=[len(p) for p in paths]
    _ind=np.argsort(path_lengths)[:k]
    paths=[paths[i] for i in _ind]

    reactions=[]
    for path in paths:
        for s,t in zip(path[:-1],path[1:]):
            rs=g[s][t]['reaction'].split(',')
            reactions.extend(rs)
    return reactions,paths


In [13]:
def find_kos(c,level, subset=None):
    try:
        if subset is None:
            subset=[]
            for l in os.listdir(os.path.join(DIR_kegg, level)):
                try:
                    cs=pd.read_csv(os.path.join(DIR_kegg, level, l,'compounds.txt'),header=None).values.flatten()
                    if c in cs:
                        subset.append(l)
                except FileNotFoundError:
                    continue
        kos=[]
        for l in subset:
            try:
                kos.extend(pd.read_csv(os.path.join(DIR_kegg, level, l,'kos.txt'),header=None).values.flatten()) 
            except FileNotFoundError:
                continue
        return np.unique(kos)
    except Exception as e:
        print("Error: ",c,level)
        return None


In [14]:
g_directed=pd.read_pickle(os.path.join(DIR_kegg,"graphs",'reaction_graph_directed.pk'))
g_undirected=pd.read_pickle(os.path.join(DIR_kegg,"graphs",'reaction_graph_undirected.pk'))
# endpoints=pd.read_csv(os.path.join(DIR_kegg, 'endpoint_compounds.txt'),sep='\t',header=None).iloc[:,0].values
def cal_compounnd_ko_mapping(carbon_names, compound_ids, endpoints):
    df=[]
    for c in tqdm(compound_ids):
        out={'compound_id':c}

        out['by_pathway']=find_kos(c,'pathways')
        # out['by_module']=find_kos(c,'modules')
        out['by_reaction']=find_kos(c,'reactions')

        for k in [1,2,3]:
            # KOs by shorted path, directed
            try:
                reactions,paths=find_reactions(g_directed, c, endpoints, k=k)
                out[f'by_k{k}_path_directed']=find_kos(c,'reactions', subset=reactions)
            except Exception as e:
                print("Error: directed,",c,k)

            # KOs by shorted path, undirected
            try:
                reactions,paths=find_reactions(g_undirected, c, endpoints, k=k)
                out[f'by_k{k}_path_undirected']=find_kos(c,'reactions', subset=reactions)
            except Exception as e:
                print("Error: undirected,",c,k)

        df.append(out)

    df=pd.DataFrame(df)
    df['carbon_names']=carbon_names
    df.set_index('carbon_names',inplace=True)
    return df

In [16]:
for dataset_name in ['zeqian','matti','bacdive']:
    df=pd.read_csv(os.path.join(DIR_kegg, f'{dataset_name}_compounds.txt'),sep=',',header=None)
    carbon_names=df.iloc[:,0].values
    compound_ids=df.iloc[:,1].values

    endpoints=pd.read_csv(os.path.join(DIR_kegg, 'endpoint_compounds.txt'),sep='\t',header=None).iloc[:,0].values
    kos=cal_compounnd_ko_mapping(carbon_names, compound_ids,endpoints)
    kos.to_pickle(f"{DIR_kegg}/selected_kos/{dataset_name}.pk")
    kos.to_csv(f"{DIR_kegg}/selected_kos/{dataset_name}.csv")

    endpoints_tca=pd.read_csv(os.path.join(DIR_kegg, 'endpoint_compounds_with_tca.txt'),sep='\t',header=None).iloc[:,0].values
    kos=cal_compounnd_ko_mapping(carbon_names, compound_ids,endpoints_tca)
    kos.to_pickle(f"{DIR_kegg}/selected_kos/{dataset_name}_tca.pk")
    kos.to_csv(f"{DIR_kegg}/selected_kos/{dataset_name}_tca.csv")

 85%|████████▌ | 85/100 [07:42<01:20,  5.40s/it]

Error: directed, nan 1
Error: undirected, nan 1
Error: directed, nan 2
Error: undirected, nan 2
Error: directed, nan 3
Error: undirected, nan 3


 86%|████████▌ | 86/100 [07:47<01:15,  5.38s/it]

Error: directed, nan 1
Error: undirected, nan 1
Error: directed, nan 2
Error: undirected, nan 2


 87%|████████▋ | 87/100 [07:53<01:10,  5.43s/it]

Error: directed, nan 3
Error: undirected, nan 3


... (11 outputs left out)

Error: directed, nan 1
Error: undirected, nan 1
Error: directed, nan 2
Error: undirected, nan 2
Error: directed, nan 3


 97%|█████████▋| 56/58 [05:02<00:10,  5.35s/it]

Error: undirected, nan 3


100%|██████████| 58/58 [05:13<00:00,  5.40s/it]


### Module related

In [50]:
# edges=pd.read_csv(os.path.join(DIR_kegg, "graphs",'module_graph_raw.txt'), sep='\t', header=None,names=['reaction','c1','c2'])
# edges=edges[(~edges['c1'].isin(ignore_compounds)) & (~edges['c2'].isin(ignore_compounds))]
# edges=edges.drop_duplicates()
# edges['c1_c2']=edges['c1']+'_'+edges['c2']
# edges['c1_c2_undirected']=edges[['c1','c2']].apply(lambda x: '_'.join(sorted(x)), axis=1)

# edges_directed=edges.groupby('c1_c2')['reaction'].apply(lambda x: ','.join(x)).reset_index()
# edges_directed['c1']=edges_directed['c1_c2'].apply(lambda x: x.split('_')[0])
# edges_directed['c2']=edges_directed['c1_c2'].apply(lambda x: x.split('_')[1])
# edges_directed['n_reactions']=edges_directed['reaction'].apply(lambda x: len(x.split(',')))


# edges_undirected=edges.groupby('c1_c2_undirected')['reaction'].apply(lambda x: ','.join(x)).reset_index()
# edges_undirected['c1']=edges_undirected['c1_c2_undirected'].apply(lambda x: x.split('_')[0])
# edges_undirected['c2']=edges_undirected['c1_c2_undirected'].apply(lambda x: x.split('_')[1])
# edges_undirected['n_reactions']=edges_undirected['reaction'].apply(lambda x: len(x.split(',')))

# g_directed=nx.from_pandas_edgelist(edges_directed, source='c1',target='c2',edge_attr=['reaction'],create_using=nx.DiGraph)

# with open(os.path.join(DIR_kegg,'module_graph_directed.pk'),'wb') as f:
#     pickle.dump(g_directed,f)

# g_undirected=nx.from_pandas_edgelist(edges_undirected, source='c1',target='c2',edge_attr=['reaction'],create_using=nx.Graph)

# with open(os.path.join(DIR_kegg,"graphs",'module_graph_undirected.pk'),'wb') as f:
#     pickle.dump(g_undirected,f)

In [51]:
# # modules
# #modules=Kegg.download_csv(["link/module",*pathways],labels=['pathway','module'])
# modules=pd.read_csv('http://rest.kegg.jp/list/module',sep='\t+',header=None,names=['module','module_name'],engine='python')
# # modules['pathway']=modules['pathway'].str.replace("path:","")
# modules['module']=modules['module'].str.replace("md:","")

# for _, module,module_name in modules.itertuples():
#     if not os.path.exists(os.path.join(DIR_kegg,"modules",module)):
#         os.makedirs(os.path.join(DIR_kegg,"modules",module))
    
#     if not os.path.exists(os.path.join(DIR_kegg,"pathways",pathway)):
#         os.makedirs(os.path.join(DIR_kegg,"pathways",pathway))
#     with open(os.path.join(DIR_kegg,"pathways",pathway,'modules.txt'),'a') as f:
#         f.write(module)
#         f.write('\n')

In [45]:
# modules
#modules=Kegg.download_csv(["link/module",*pathways],labels=['pathway','module'])
modules=pd.read_csv('http://rest.kegg.jp/list/module',sep='\t+',header=None,names=['module','module_name'],engine='python')
# modules['pathway']=modules['pathway'].str.replace("path:","")
modules['module']=modules['module'].str.replace("md:","")

for _, module,__ in modules.itertuples():
    if not os.path.exists(os.path.join(DIR_kegg,"modules",module)):
        os.makedirs(os.path.join(DIR_kegg,"modules",module))
    
    # if not os.path.exists(os.path.join(DIR_kegg,"pathways",pathway)):
    #     os.makedirs(os.path.join(DIR_kegg,"pathways",pathway))
    # with open(os.path.join(DIR_kegg,"pathways",pathway,'modules.txt'),'a') as f:
    #     f.write(module)
    #     f.write('\n')

In [46]:
modules=os.listdir(os.path.join(DIR_kegg,"modules"))

compounds=Kegg.download_csv(["link/compound",*modules],labels=['module','compound'])
compounds['module']=compounds['module'].str.replace("md:","")
compounds['compound']=compounds['compound'].str.replace("cpd:","")

for _, module, compound in compounds.itertuples():
    with open(os.path.join(DIR_kegg,"modules",module,'compounds.txt'),'a') as f:
        f.write(compound)
        f.write('\n')

kos=Kegg.download_csv(["link/ko",*modules],labels=['module','ko'])
kos['module']=kos['module'].str.replace("md:","")
kos['ko']=kos['ko'].str.replace("ko:","")

for _, module, ko in kos.itertuples():
    with open(os.path.join(DIR_kegg,"modules",module,'kos.txt'),'a') as f:
        f.write(ko)
        f.write('\n')

reactions=Kegg.download_csv(["link/reaction",*modules],labels=['module','reaction'])
reactions['module']=reactions['module'].str.replace("md:","")
reactions['reaction']=reactions['reaction'].str.replace("rn:","")

for _, module, reaction in reactions.itertuples():
    with open(os.path.join(DIR_kegg,"modules",module,'reactions.txt'),'a') as f:
        f.write(reaction)
        f.write('\n')

49it [01:28,  1.81s/it]
49it [01:27,  1.79s/it]
49it [01:27,  1.78s/it]


In [47]:
# modules

modules=os.listdir(os.path.join(DIR_kegg,"modules"))
for module in tqdm(modules, desc="Retriving module information"):
    try:
        res=requests.get(f"http://rest.kegg.jp/get/{module}")
        if not bool(res):
            raise ValueError
        with open(os.path.join(DIR_kegg,"modules",module,module),'w') as f:
            f.write(res.text)
    except Exception as e:
        print("Failed: ",module)
        continue

Retriving module information:   0%|          | 0/481 [00:00<?, ?it/s]

Retriving module information: 100%|██████████| 481/481 [13:47<00:00,  1.72s/it]


In [48]:

def parse_module(text):
    out=[]
    text=re.findall(r"REACTION\s+([RC\d,->\n\s\+]+)\nCOMPOUND",text)
    if text:
        text=text[0].strip()
    else:
        print('Failed')
        return out
    for line in text.split("\n"):
        line=line.strip()
        if len(line)==0:
            continue
        rs=re.findall('[R\d, ]+',line)[0]
        lefts=re.findall('[C\d, \+]+->',line)[0]
        rights=re.findall('->[C\d, \+]+',line)[0]

        rs=[re.findall('R\d+',r)[0] for r in re.split(r'[\+,]',rs.strip()) if re.findall('R\d+',r)]
        lefts=[re.findall('C\d+',l)[0] for l in lefts.strip().split('+') if re.findall('C\d+',l)]
        rights=[re.findall('C\d+',r)[0] for r in rights.strip().split('+') if re.findall('C\d+',r)]
        out.append((rs,lefts,rights))
    return out


In [49]:
# # module graph

# with open(os.path.join(DIR_kegg, 'module_graph_raw.txt'),'w') as f_graph:
#     for module in tqdm(os.listdir(os.path.join(DIR_kegg,'modules')), desc="Building module graph..."):
#         try:
#             with open(os.path.join(DIR_kegg,"modules",module,module),'r') as f:
#                 text=f.read()
#             for rs, lefts, rights in parse_module(text):
#                 for left in lefts:
#                     for right in rights:
#                         f_graph.write(f"{','.join(rs)}\t{left}\t{right}\n")
#         except Exception as e:
#             print("Error parsing ", module)
#             raise e
